# Gradient Descent for Linear Regression — Boston Housing Dataset (Built from Scratch)

This notebook walks through a **complete from-scratch implementation of gradient descent** using the `rice_ml` package.

We use gradient descent to train a **linear regression model** on the Boston Housing dataset — a widely used benchmark for regression problems.

Throughout this notebook, we will:

- Load and explore the Boston Housing dataset
- Conduct exploratory data analysis (EDA)
- Apply feature standardization with our own preprocessing tools
- Fit a linear regression model via gradient descent
- Assess model performance with regression metrics
- Plot the convergence curve and residual patterns

No external ML libraries (e.g. sklearn) are used at any point.

## About the Boston Housing Dataset

This dataset captures housing information across suburbs of Boston, Massachusetts.

Every row corresponds to a census tract and includes **13 numerical features** that describe socioeconomic and environmental conditions.

### Features

| Feature | Description |
|-------|-------------|
| CRIM | Per capita crime rate by town |
| ZN | Fraction of residential land zoned for large lots |
| INDUS | Proportion of non-retail business acres |
| CHAS | Charles River binary variable (1 if tract borders river) |
| NOX | Nitrogen oxides concentration |
| RM | Average number of rooms per dwelling |
| AGE | Proportion of owner-occupied units built before 1940 |
| DIS | Weighted distance to employment centers |
| RAD | Accessibility index to radial highways |
| TAX | Full-value property tax rate per $10,000 |
| PTRATIO | Pupil-to-teacher ratio |
| B | Proportion related to Black population |
| LSTAT | Percentage of lower-status population |

### Target

- **MEDV** — Median value of owner-occupied homes (in \$1000s)

All variables are numeric with no missing values.

## Imports and Setup

We import NumPy, Matplotlib, and Pandas, along with custom functions from `preprocessing.py` and `post_processing.py`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd

# Import preprocessing utilities
from rice_ml.processing.preprocessing import minmax_scale, standardize, train_test_split
from rice_ml.processing.post_processing import r2_score

# Import Gradient Descent classes
from rice_ml.supervised_learning.gradient_descent import (
    GradientDescent1D, GradientDescentND
)

## Loading the Data

We fetch the Boston Housing dataset directly from the UCI Machine Learning Repository.
Since the file is whitespace-delimited and has no header row, we manually supply the column names.

In [ ]:
# Upload Dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/housing/housing.data"
column_names = [
    "CRIM",
    "ZN",
    "INDUS",
    "CHAS",
    "NOX",
    "RM",
    "AGE",
    "DIS",
    "RAD",
    "TAX",
    "PTRATIO",
    "B",
    "LSTAT",
    "MEDV",
]

import pandas as pd
df = pd.read_csv(url, sep='\s+', names=column_names)

df.head()

## Exploratory Data Analysis (EDA)

Before building our model, we examine the dataset to gain a better understanding of:

- How each feature is distributed
- The shape of the target variable
- How feature magnitudes differ across columns

This analysis helps us justify the need for feature standardization before applying gradient descent.

### Distribution of the Target Variable

We begin by plotting the distribution of home prices (MEDV).

### Feature Scale Variation

Gradient descent is sensitive to differences in feature scale.  
The plots below reveal that features differ substantially in magnitude, which motivates standardization.

## Comparing Feature Variances

We also compute feature variances numerically to quantify these scale differences.

In [ ]:
# Target variable distribution:

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 5))
sns.histplot(df["MEDV"], bins=30, kde=True)
plt.xlabel("Median Home Value (MEDV)")
plt.ylabel("Frequency")
plt.title("Distribution of Target Variable (MEDV)")
plt.show()

plt.figure(figsize=(6, 2))
sns.boxplot(x=df["MEDV"])
plt.title("Boxplot of MEDV")
plt.show()

# Feature scale differences

X = df.drop(columns=["MEDV"])

plt.figure(figsize=(12, 6))
sns.boxplot(data=X, orient="h")
plt.title("Feature Scale Comparison (Before Standardization)")
plt.xlabel("Feature Value")
plt.show()

# Feature variance comparison

import numpy as np

variances = X.var()

plt.figure(figsize=(10, 4))
variances.plot(kind="bar")
plt.ylabel("Variance")
plt.title("Feature Variance Comparison")
plt.show()

### EDA Summary

The Boston Housing dataset is entirely numeric with no missing entries.
The target variable (`MEDV`) follows an approximately bell-shaped distribution with moderate right skew and a few high-value outliers.

Feature scales vary widely — `TAX`, `RAD`, and `CRIM` exhibit far larger magnitudes than other predictors. This confirms the need for feature standardization prior to gradient descent optimization.

## Data Preprocessing

We split the dataset into:

- Feature matrix $X \in \mathbb{R}^{n \times d}$
- Target vector $y \in \mathbb{R}^{n}$

We then **standardize** all features using:

$$
X_{\text{std}} = \frac{X - \mu}{\sigma}
$$

Standardization brings all features to a comparable scale, which is essential for stable convergence in gradient descent.

Finally, we partition the data into training and test sets using our custom `train_test_split` function.

In [ ]:
X = df.drop(columns=["MEDV"]).values
y = df["MEDV"].values.reshape(-1, 1)

X.shape, y.shape

# Preprocessing: Standardization
X_scaled = standardize(X)
y_scaled = standardize(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

### Adding the Bias Term

To allow the model to learn an intercept, we augment the feature matrix with a column of ones:

$$
\tilde{X} =
\begin{bmatrix}
1 & x_{11} & \dots \\
1 & x_{21} & \dots \\
\vdots & \vdots & \ddots
\end{bmatrix}
$$

The weight associated with this column represents the bias term $b$, enabling the model to make predictions even when all features are zero.

In [ ]:
# Add bias column
def add_bias(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

X_train_b = add_bias(X_train)
X_test_b  = add_bias(X_test)

## How Gradient Descent Works for Linear Regression

Gradient descent is an **iterative first-order optimization algorithm** that minimizes a differentiable objective function by repeatedly moving in the direction of steepest descent.
For linear regression, the goal is to find the weight vector $w$ that minimizes **Mean Squared Error (MSE)**.

### The Loss Function (MSE)

$$
\mathcal{L}(w) = \frac{1}{n} \sum_{i=1}^{n} \left(y_i - x_i^\top w\right)^2
$$

where:
- $x_i$ is the feature vector for sample $i$
- $y_i$ is the true label
- $n$ is the total number of training samples

This measures the average squared deviation between predictions and ground truth.

---

### Computing the Gradient

To descend the loss surface, we compute the partial derivatives with respect to $w$:

$$
\nabla_w \mathcal{L}(w) = \frac{2}{n} X^\top (Xw - y)
$$

The gradient points in the direction of greatest increase, so we move **opposite** to it to reduce loss.

---

### The Weight Update Rule

At each iteration, weights are updated as:

$$
w^{(t+1)} = w^{(t)} - \alpha \nabla_w \mathcal{L}(w)
$$

where:
- $\alpha$ is the learning rate (step size)
- $t$ is the current iteration index

This continues until the weight update magnitude falls below a convergence threshold.

---

### Design Note: Using Closures for Modularity

In our implementation, the gradient is defined as a **closure** that captures the training data $X$ and $y$ from its enclosing scope.
This allows us to expose a clean interface:

grad_f(w) → gradient vector

Benefits of this approach:
- The `GradientDescentND` class stays **completely generic** — it knows nothing about the dataset or loss
- The same optimizer can be reused for any differentiable objective
- Separation of concerns improves code clarity and reusability

In [ ]:
def make_gradient_function(X, y):
    """
    Returns a gradient function grad_f(w) for MSE linear regression.

    Parameters
    ----------
    X : ndarray, shape (n_samples, n_features)
        Design matrix (including bias column).

    y : ndarray, shape (n_samples, 1)
        Standardized target values.

    Returns
    -------
    grad_f : function
        Accepts a weight vector w and returns the MSE gradient.
    """
    n = len(X)

    def grad_f(w):
        w = w.reshape(-1, 1)              # ensure column vector
        y_pred = X @ w
        grad = (2/n) * X.T @ (y_pred - y) # MSE gradient
        return grad.flatten()             # flatten to shape (n_features,)

    return grad_f

grad_f = make_gradient_function(X_train_b, y_train)

## Training with GradientDescentND

With `grad_f(w)` defined, we pass it to `GradientDescentND`, which applies the update rule:

$$
w_{t+1} = w_t - \alpha \nabla L(w_t)
$$

Key hyperparameters:

- $\alpha$ — learning rate controlling the step size
- `tol` — convergence threshold; training stops when weight change falls below this value
- `max_iter` — maximum number of update steps

Each weight vector visited during training is stored in `history` for later visualization.

We initialize weights at zero and let the optimizer run.

In [ ]:
n_features = X_train_b.shape[1]
w0 = np.zeros(n_features)   # 1D vector as required by your class

gd = GradientDescentND(
    grad_f=grad_f,
    alpha=0.01,
    tol=1e-6,
    max_iter=3000
)

history = gd.fit(w0)

## Reconstructing the Loss Curve

`GradientDescentND` records the sequence of weight vectors but does not compute the loss automatically.
We reconstruct the training loss curve by evaluating MSE at each saved weight vector:

$$
\text{MSE}(w) = \frac{1}{n} \sum_{i=1}^{n} \left( \hat{y}_i - y_i \right)^2
$$

Inspecting the loss curve allows us to:

- Confirm that the optimizer is converging
- Diagnose learning rate issues (divergence or slow convergence)
- Verify optimization stability

A smooth, monotonically decreasing curve signals that the chosen learning rate is well-suited for this problem.

In [ ]:
def mse(X, y, w):
    w = w.reshape(-1, 1)
    return np.mean((X @ w - y)**2)

losses = [mse(X_train_b, y_train, w) for w in history]

final_w = history[-1].reshape(-1, 1)

print("Final training loss:", losses[-1])

## Plotting the Loss Curve

In [ ]:
plt.plot(losses)
plt.xlabel("Iteration")
plt.ylabel("MSE Loss")
plt.title("Gradient Descent Convergence")
plt.show()

## Evaluating on the Test Set

We assess generalization using the **coefficient of determination** ($R^2$), which measures how much of the target variance the model explains:

$$
R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}
$$

Interpretation:

- **$R^2 = 1.0$** — perfect predictions
- **$R^2 = 0.0$** — model does no better than predicting the mean
- **$R^2 < 0$** — model is worse than the mean baseline

In [ ]:
y_pred_test = X_test_b @ final_w
test_r2 = r2_score(y_test, y_pred_test)

print("Test R^2:", test_r2)

## Actual vs. Predicted Values

In [ ]:
plt.scatter(y_test, y_pred_test, alpha=0.6)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red"
)
plt.xlabel("Actual MEDV (scaled)")
plt.ylabel("Predicted MEDV (scaled)")
plt.title("Actual vs Predicted (Test Set)")
plt.show()

## Results Summary

We built a linear regression model from scratch using gradient descent on the Boston Housing dataset.
After standardizing features and targets, the optimizer converged smoothly — visible in the steadily declining MSE curve — confirming that the learning rate was appropriate and that scaling resolved the ill-conditioning caused by feature scale differences.

The trained model achieved an $R^2$ score of approximately **0.77** on held-out test data, meaning it accounts for roughly 77% of the variance in housing prices.
The scatter plot of predicted versus actual values shows a clear linear trend with moderate spread, reflecting both the explanatory power and the inherent limitations of a purely linear model on real-world housing data.

Overall, these results validate our from-scratch gradient descent implementation and demonstrate its ability to fit a meaningful predictive model when paired with correct preprocessing and well-chosen hyperparameters.